In [9]:
from pathlib import Path
from hashlib import sha256
from datetime import datetime, timezone
from uuid import uuid4
import os

from dotenv import load_dotenv
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

In [10]:
load_dotenv()
openai_api_key = os.environ.get("OPENAI_API_KEY")

In [11]:
DATA_DIR = Path("data")

In [12]:
persist_directory = DATA_DIR / "bookstore"

In [13]:
pdf_files_paths = DATA_DIR.glob("*.pdf")

In [14]:
list(pdf_files_paths)

[WindowsPath('data/Atomic habits ( PDFDrive ).pdf'),
 WindowsPath('data/attention.pdf'),
 WindowsPath('data/BhagavadGita.pdf'),
 WindowsPath('data/the-5-am-club.pdf')]

In [15]:
def normalize_name(text: str) -> str:
    return "".join(ch if ch.isalnum() else "_" for ch in text.lower()).strip("_")


def file_sha256(path: Path, chunk_size: int = 1024 * 1024) -> str:
    hasher = sha256()
    with path.open("rb") as f:
        while True:
            block = f.read(chunk_size)
            if not block:
                break
            hasher.update(block)
    return hasher.hexdigest()

In [16]:
all_documents = []
source_ids = set() 

for pdf_file_path in DATA_DIR.glob("*.pdf"):
    print(f"{pdf_file_path.name}: {file_sha256(pdf_file_path)}")
    loader = PyMuPDFLoader(str(pdf_file_path))
    documents = loader.load()

    source_checksum = file_sha256(pdf_file_path)
    source_id = normalize_name(pdf_file_path.stem)
    source_ids.add(source_id)

    for index, doc in enumerate(documents):
        doc.metadata["source"] = pdf_file_path.name
        doc.metadata["source_id"] = source_id
        doc.metadata["source_checksum"] = source_checksum
        doc.metadata["page_number"] = index + 1
        doc.metadata["chunk_id"] = str(uuid4())
        doc.metadata["chunk_index"] = index
        doc.metadata["chunk_size"] = len(doc.page_content)
        doc.metadata["created_at"] = datetime.now(timezone.utc).isoformat()

    all_documents.extend(documents)

documents = all_documents
print(len(documents), "documents loaded from all PDFs.")
sorted(set(source_ids))

Atomic habits ( PDFDrive ).pdf: a8cacf49ea808daacbef95466b3bcbd3db371d63fdeabcb8d26cf15504062878
attention.pdf: bdfaa68d8984f0dc02beaca527b76f207d99b666d31d1da728ee0728182df697
BhagavadGita.pdf: ff112b0b056d303b792f6f2e68cbd73a89adf612fa9113f932446cdea7741583
the-5-am-club.pdf: 089bb3fc5b41e7de713444b93214434e1d887c47dff66e8e4ae075aeed89440b
1475 documents loaded from all PDFs.


['atomic_habits___pdfdrive', 'attention', 'bhagavadgita', 'the_5_am_club']

In [17]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150,
)

In [18]:
chunks = splitter.split_documents(documents)
chunks[0].metadata

{'producer': 'calibre 3.48.0 [https://calibre-ebook.com]',
 'creator': 'calibre 3.48.0 [https://calibre-ebook.com]',
 'creationdate': '2020-04-30T18:46:22+00:00',
 'source': 'Atomic habits ( PDFDrive ).pdf',
 'file_path': 'data\\Atomic habits ( PDFDrive ).pdf',
 'total_pages': 256,
 'format': 'PDF 1.4',
 'title': 'Atomic habits \\( PDFDrive.com \\).pdf',
 'author': 'James Clear',
 'subject': '',
 'keywords': '',
 'moddate': '',
 'trapped': '',
 'modDate': '',
 'creationDate': "D:20200430184622+00'00'",
 'page': 2,
 'source_id': 'atomic_habits___pdfdrive',
 'source_checksum': 'a8cacf49ea808daacbef95466b3bcbd3db371d63fdeabcb8d26cf15504062878',
 'page_number': 3,
 'chunk_id': '1a05861e-8fe2-4d81-bafa-cc39a2185156',
 'chunk_index': 2,
 'chunk_size': 531,
 'created_at': '2026-07-29T00:17:51.840494+00:00'}

In [19]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

In [20]:
pdf_paths = sorted(DATA_DIR.glob("*.pdf"))
collection_names = [normalize_name(p.stem) for p in pdf_paths]
collection_names

['atomic_habits___pdfdrive', 'attention', 'bhagavadgita', 'the_5_am_club']

In [21]:
collection_store = {}

for collection in collection_names:
    collection_path = persist_directory / collection

    if not collection_path.exists():
        collection_path.mkdir(parents=True, exist_ok=True)

    temp_store = Chroma(
        persist_directory=str(collection_path),
        embedding_function=embeddings,
        collection_name=collection,
    )

    try:
        temp_store._client.delete_collection(name=collection)
    except Exception:
        pass

    collection_store[collection] = Chroma(
        persist_directory=str(collection_path),
        embedding_function=embeddings,
        collection_name=collection,
    )

collection_store

{'atomic_habits___pdfdrive': <langchain_chroma.vectorstores.Chroma at 0x1b9573afb60>,
 'attention': <langchain_chroma.vectorstores.Chroma at 0x1b953ff4500>,
 'bhagavadgita': <langchain_chroma.vectorstores.Chroma at 0x1b9574f5880>,
 'the_5_am_club': <langchain_chroma.vectorstores.Chroma at 0x1b957906000>}

In [22]:
missing_source_ids = set()
inserted_count = 0
batch_size = 100

chunks_by_source = {}
for chunk in chunks:
    source_id = chunk.metadata["source_id"]
    if source_id not in chunks_by_source:
        chunks_by_source[source_id] = []
    chunks_by_source[source_id].append(chunk)

for source_id, source_chunks in chunks_by_source.items():
    vector_store = collection_store.get(source_id)
    if not vector_store:
        missing_source_ids.add(source_id)
        continue

    for start in range(0, len(source_chunks), batch_size):
        end = start + batch_size
        vector_store.add_documents(source_chunks[start:end])
        inserted_count += len(source_chunks[start:end])

print("Inserted chunks:", inserted_count)
print("Missing source_id keys:", sorted(missing_source_ids))

Inserted chunks: 3583
Missing source_id keys: []


# Creating retriever for the RAG system

In [23]:
query = ['What does krishna says about the Mahabharat battle']

In [24]:
collection_store

{'atomic_habits___pdfdrive': <langchain_chroma.vectorstores.Chroma at 0x1b9573afb60>,
 'attention': <langchain_chroma.vectorstores.Chroma at 0x1b953ff4500>,
 'bhagavadgita': <langchain_chroma.vectorstores.Chroma at 0x1b9574f5880>,
 'the_5_am_club': <langchain_chroma.vectorstores.Chroma at 0x1b957906000>}

In [25]:
bg_vectorstore = collection_store.get('bhagavadgita')
bg_vectorstore

In [64]:
bg_retriver = bg_vectorstore.as_retriever(search_kwargs={"k": 10})

In [27]:
bg_retriver.invoke(input=query[0])

[Document(id='8a350e3b-0e4a-4c09-a750-4954c92e9481', metadata={'creator': 'Pages', 'source_checksum': 'ff112b0b056d303b792f6f2e68cbd73a89adf612fa9113f932446cdea7741583', 'page': 50, 'total_pages': 952, 'chunk_id': 'fa4cc58d-ffa0-445c-987b-b7d7dd700bef', 'title': 'Bhagavad-gita As It Is with pics!', 'format': 'PDF 1.3', 'page_number': 51, 'source_id': 'bhagavadgita', 'chunk_size': 1333, 'modDate': "D:20120508142201Z00'00'", 'creationDate': "D:20120508142201Z00'00'", 'chunk_index': 50, 'created_at': '2026-07-29T00:17:54.661948+00:00', 'author': 'me', 'moddate': "D:20120508142201Z00'00'", 'keywords': '', 'producer': 'Mac OS X 10.7.3 Quartz PDFContext', 'source': 'BhagavadGita.pdf', 'creationdate': "D:20120508142201Z00'00'", 'trapped': '', 'subject': '', 'file_path': 'data\\BhagavadGita.pdf'}, page_content='position as a lion. Indirectly, by the symbolism of the conchshell, he informed \nhis depressed grandson Duryodhana that he had no chance of victory in the \nbattle, because the Supreme

# Building a Part 2 - RAG Pipeline

## Creating Stuff Documents Chain

In [28]:
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

In [29]:
from langchain_core.prompts import ChatPromptTemplate

In [30]:
system_prompt = '''

You are an assistant that answers questions based on the context provided. You are given a question and a set of documents that may contain the answer.
Your task is to provide a concise and accurate answer to the question using the information from the documents. 
If the answer is not present in the documents, respond with "I don't know."

context:{context}

'''

In [31]:
prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}"),
])

In [32]:
from langchain.chat_models.base import init_chat_model

In [33]:
llm = init_chat_model(
    model='gpt-5.4-nano',
    api_key=os.environ.get("OPENAI_API_KEY"),
    temperature=0.0,
    max_tokens=512,
) 

In [34]:
response = llm.invoke("Hi")
response.content


'Hi! 👋 How can I help you today?'

In [35]:
combine_docs_chain = create_stuff_documents_chain(
    llm, prompt
)

## Creating Retrival Chain

In [36]:
from langchain_classic.chains import create_retrieval_chain

In [37]:
rag_chain = create_retrieval_chain(bg_retriver, combine_docs_chain)

In [38]:
result = rag_chain.invoke({'input': 'What did Krishna say about the Mahabharat battle?'})
result['answer']

'From the provided text, Krishna said that **where Krishna and Arjuna are present, there will be all good fortune**, so **victory is certain for Arjuna’s side**. It also states that Krishna was personally present as Arjuna’s charioteer and that **Arjuna was not really fighting on his own, but was carrying out Krishna’s orders in full Krishna consciousness**, so he wouldn’t be entangled in the reactions of work.'

# Adding Hybrid Search

In [39]:
bg_retriver #This is dense retriever. This does a semantic search on the vector store created from the Bhagavad Gita PDF. It retrieves the top 5 relevant documents based on the query provided.

VectorStoreRetriever(tags=['Chroma', 'OpenAIEmbeddings'], vectorstore=<langchain_chroma.vectorstores.Chroma object at 0x000001B9574F5880>, search_kwargs={'k': 5})

### Adding Dense + Sparse to Build Hybrid Search

In [40]:
from langchain_community.retrievers import BM25Retriever
from langchain_core.documents import Document


In [41]:
chunks[0]

Document(metadata={'producer': 'calibre 3.48.0 [https://calibre-ebook.com]', 'creator': 'calibre 3.48.0 [https://calibre-ebook.com]', 'creationdate': '2020-04-30T18:46:22+00:00', 'source': 'Atomic habits ( PDFDrive ).pdf', 'file_path': 'data\\Atomic habits ( PDFDrive ).pdf', 'total_pages': 256, 'format': 'PDF 1.4', 'title': 'Atomic habits \\( PDFDrive.com \\).pdf', 'author': 'James Clear', 'subject': '', 'keywords': '', 'moddate': '', 'trapped': '', 'modDate': '', 'creationDate': "D:20200430184622+00'00'", 'page': 2, 'source_id': 'atomic_habits___pdfdrive', 'source_checksum': 'a8cacf49ea808daacbef95466b3bcbd3db371d63fdeabcb8d26cf15504062878', 'page_number': 3, 'chunk_id': '1a05861e-8fe2-4d81-bafa-cc39a2185156', 'chunk_index': 2, 'chunk_size': 531, 'created_at': '2026-07-29T00:17:51.840494+00:00'}, page_content='AN IMPRINT OF PENGUIN RANDOM HOUSE LLC\n375 Hudson Street\nNew York, New York 10014\nCopyright © 2018 by James Clear\nPenguin supports copyright. Copyright fuels creativity, enc

In [46]:
sparse_retriver = BM25Retriever.from_documents(chunks, k=5)
sparse_retriver

BM25Retriever(vectorizer=<rank_bm25.BM25Okapi object at 0x000001B958FB0470>, k=5)

### Why combine dense and sparse retrieval?

- Dense retrieval uses embeddings to find chunks that are semantically similar to the query.
- Sparse retrieval with BM25 looks for exact or near-exact keyword overlap.
- A hybrid retriever combines both so the search is stronger on paraphrased questions and on questions with important names, terms, or dates.
- In this notebook, the two retrievers are given equal weight first so neither side dominates.
- What to observe: the hybrid retriever should return a more balanced set of chunks than either retriever alone.

In [51]:
from langchain_classic.retrievers import EnsembleRetriever

In [70]:
hybrid_retriever=EnsembleRetriever(
    retrievers=[bg_retriver,sparse_retriver],
    weights=[0.5,0.5]
)
hybrid_retriever

EnsembleRetriever(retrievers=[VectorStoreRetriever(tags=['Chroma', 'OpenAIEmbeddings'], vectorstore=<langchain_chroma.vectorstores.Chroma object at 0x000001B9574F5880>, search_kwargs={'k': 10}), BM25Retriever(vectorizer=<rank_bm25.BM25Okapi object at 0x000001B958FB0470>, k=5)], weights=[0.5, 0.5])

### How to read the hybrid retriever output

- `weights=[0.5, 0.5]` means both retrievers contribute equally to the final ranking.
- Raise the dense weight when the query is more conceptual or paraphrased.
- Raise the sparse weight when exact words, names, or technical terms matter most.
- The output documents are merged and ranked together before they are passed into the RAG chain.
- What to observe: compare the `doc_res` from `bg_retriver` with the `doc_res` from `hybrid_retriever` to see how the candidate chunks change.

In [65]:
doc_res = bg_retriver.invoke('input=What did Krishna say about the Mahabharat battle?')

In [66]:
for doc in doc_res:
    print(f"Source: {doc.metadata['source']}, Chunk index: {doc.metadata['chunk_index']}")

Source: BhagavadGita.pdf, Chunk index: 50
Source: BhagavadGita.pdf, Chunk index: 301
Source: BhagavadGita.pdf, Chunk index: 901
Source: BhagavadGita.pdf, Chunk index: 901
Source: BhagavadGita.pdf, Chunk index: 128
Source: BhagavadGita.pdf, Chunk index: 39
Source: BhagavadGita.pdf, Chunk index: 89
Source: BhagavadGita.pdf, Chunk index: 896
Source: BhagavadGita.pdf, Chunk index: 602
Source: BhagavadGita.pdf, Chunk index: 94


In [71]:
doc_res = hybrid_retriever.invoke('input=What did Krishna say about the Mahabharat battle?')

In [72]:
for doc in doc_res:
    print(f"Source: {doc.metadata['source']}, Chunk index: {doc.metadata['chunk_index']}")

Source: BhagavadGita.pdf, Chunk index: 50
Source: BhagavadGita.pdf, Chunk index: 911
Source: BhagavadGita.pdf, Chunk index: 301
Source: BhagavadGita.pdf, Chunk index: 910
Source: BhagavadGita.pdf, Chunk index: 901
Source: BhagavadGita.pdf, Chunk index: 940
Source: BhagavadGita.pdf, Chunk index: 901
Source: BhagavadGita.pdf, Chunk index: 903
Source: BhagavadGita.pdf, Chunk index: 128
Source: BhagavadGita.pdf, Chunk index: 951
Source: BhagavadGita.pdf, Chunk index: 39
Source: BhagavadGita.pdf, Chunk index: 89
Source: BhagavadGita.pdf, Chunk index: 896
Source: BhagavadGita.pdf, Chunk index: 602
Source: BhagavadGita.pdf, Chunk index: 94


In [73]:
llm

ChatOpenAI(metadata={'lc_versions': {'langchain-core': '1.4.9', 'langchain': '1.3.14', 'langchain-openai': '1.3.5'}}, output_version=None, profile={'name': 'GPT-5.4 nano', 'release_date': '2026-03-17', 'last_updated': '2026-03-17', 'open_weights': False, 'max_input_tokens': 400000, 'max_output_tokens': 128000, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': False, 'image_url_inputs': True, 'pdf_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True, 'tool_call_streaming': True}, client=<openai.resources.chat.completions.completions.Completions object at 0x000001B9594F7F20>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x000001B9594F7AD0>, root_client=<openai.OpenAI objec

In [75]:
combine_docs_chain

RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableLambda(format_docs)
}), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
| ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template='\n\nYou are an assistant that answers questions based on the context provided. You are given a question and a set of documents that may contain the answer.\nYour task is to provide a concise and accurate answer to the question using the information from the documents. \nIf the answer is not present in the documents, respond with "I don\'t know."\n\ncontext:{context}\n\n'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs={})])
| ChatOpenAI(metadata={'lc_

In [76]:
hybride_retriver_chain =  create_retrieval_chain(hybrid_retriever, combine_docs_chain)

In [77]:
response = hybride_retriver_chain.invoke({"input": "What did Krishna say about the Mahabharat battle?"})

In [78]:
response['answer']

'The document says that when Arjuna was reluctant to fight, **Krishna explained that “Time” is destroying everything and that He had come to engage (destroy) all people**. Specifically, Krishna said:\n\n- **“Time I am, destroyer of the worlds”**\n- **“I have come to engage all people”**\n- **“With the exception of you [the Pāṇḍavas], all the soldiers here on both sides will be slain.”**'